# Showcase: Building a LINDAS Cube with `pylindas`

This is the recommended starting point for `pylindas`. It turns the small tabular dataset in `mock/data.csv` into an RDF **data cube** using [`pylindas`](https://github.com/Kronmar-Bafu/lindas-pylindas), following the [`cube.link`](https://cube.link) schema. It walks through the complete process end to end so that every step is inspectable, and explains what each piece of the cube schema means along the way.

The non-notebook equivalent of this walkthrough is `mock/mock.py`, which does the same thing in one script.

## What is a `cube.link` cube?

[`cube.link`](https://cube.link) is a vocabulary for publishing statistical data as RDF. It describes a dataset as a **cube** made of three kinds of resource:

- **`cube:Cube`** — the dataset itself, carrying metadata (title, publisher, version, status, ...).
- **`cube:Observation`** — one data point (one row of the table). Every observation belongs to the cube's **`cube:ObservationSet`**.
- **`cube:ObservationConstraint`** (the *shape*) — a SHACL description of the dimensions: their types, datatypes, allowed values, and units. It is what makes the cube self-describing and validatable.

Each column of the source table becomes a **dimension**. `cube.link` distinguishes a few dimension roles:

| Role | Meaning |
|------|---------|
| **Key Dimension** | Identifies the observation (the "coordinates" of the data point), e.g. year and station. |
| **Measure Dimension** | The measured value itself, e.g. the recorded quantity. |
| **Standard Error** | An uncertainty that `relates-to` a measure. |
| **Annotation** | Extra context about a row, e.g. a publication status. |

**Steps in this notebook**
1. Install dependencies
2. Load the data
3. Load and read the cube description (`description.yml`)
4. Initialize the `Cube` object
5. Prepare the data (apply the mappings)
6. Write the `cube:Cube`, observations and shape
7. Serialize and inspect the Turtle
8. (Optional) validate against the `cube.link` SHACL profiles
9. (Optional) upload to a triple store

## 0. Install dependencies

Run once per kernel. If you already have `pylindas` installed in the active environment you can skip this cell. Installing `pylindas` pulls in `pandas`, `rdflib` and the rest of its dependencies.

In [ ]:
%pip install -q pandas pyyaml "pylindas~=0.5.0"

## 1. Load the data

The dataset lives in `mock/data.csv` next to this notebook. It is a simple yearly time series: a measured value and its standard error, per station, with a publication status. One row = one observation.

| column | role in the cube |
|--------|------------------|
| `Jahr` | Key Dimension (year) |
| `Station` | Key Dimension (measuring station) |
| `Wert` | Measure Dimension (the value) |
| `Standardfehler` | Standard Error (relates to `Wert`) |
| `Status` | Annotation (publication status) |

`Jahr` + `Station` together uniquely identify each row — those are the cube's keys.

In [ ]:
import os
import pandas as pd

# Locate the mock/ data folder robustly, whether the kernel starts in
# example/Cubes/ (next to this notebook) or at the repository root.
_candidates = [
    os.path.join(os.path.abspath(""), "mock"),
    os.path.join(os.path.abspath(""), "example", "Cubes", "mock"),
]
BASEDIR = next((p for p in _candidates if os.path.isdir(p)), _candidates[0])

DATA_PATH = os.path.join(BASEDIR, "data.csv")
CONFIG_PATH = os.path.join(BASEDIR, "description.yml")

df = pd.read_csv(DATA_PATH)

print(f"Loaded {len(df)} rows from {DATA_PATH}")
print("Years    :", df['Jahr'].min(), "-", df['Jahr'].max())
print("Stations :", sorted(df['Station'].unique()))
print("Statuses :", sorted(df['Status'].unique()))
df.head()

## 2. Load and read the cube description

`mock/description.yml` is the bridge between the table and the `cube.link` schema. It has two parts:

**Cube-level metadata** — becomes properties of the `cube:Cube` resource:
- `Name` / `Description` — multilingual title and description (`schema:name`, `schema:description`).
- `Publisher` / `Creator` / `Contributor` — responsible organisations (identified by IRI).
- `Base-URI`, `Identifier`, `Version` — together these mint the cube's URI, e.g. `{Base-URI}{Identifier}/{Version}`.
- `Work Status` — `Draft` or `Published` (`schema:creativeWorkStatus`).
- `Visualize: True` — flags the cube for the LINDAS [visualize](https://ld.admin.ch/application/visualize) tool, which adds extra shape requirements.
- `Accrual Periodicity` — how often the data is updated.

**`dimensions`** — one entry per table column, declaring its role and how to turn the raw cell value into RDF. Key fields:
- `dimension-type` — Key / Measure / Standard Error / Annotation (see the intro table).
- `scale-type` — the statistical scale: `nominal`, `ordinal`, `interval`, `ratio`.
- `datatype` — the literal type (`float`, `string`) or `URI` when the value is turned into a resource.
- `path` — the short name used in the property URI for that dimension.
- `mapping` — how to transform the cell value (see step 4).
- `unit` — a [QUDT](http://qudt.org) unit for measures (`KiloGM`, `PERCENT`, ...).
- `data-kind` — extra semantics such as `temporal` for a year.

In [ ]:
import yaml

with open(CONFIG_PATH, encoding="utf-8") as f:
    cube_yaml = yaml.safe_load(f)

print("Cube name   :", cube_yaml["Name"]["en"])
print("Base-URI    :", cube_yaml["Base-URI"])
print("Identifier  :", cube_yaml["Identifier"], "  Version:", cube_yaml["Version"])
print("Work Status :", cube_yaml["Work Status"])
print()
print("Dimensions and their roles:")
for name, d in cube_yaml["dimensions"].items():
    unit = f"  unit={d['unit']}" if "unit" in d else ""
    print(f"  {name:15s} {d['dimension-type']:18s} scale={d['scale-type']:8s} datatype={d['datatype']}{unit}")

## 3. Initialize the `Cube` object

`Cube(dataframe, cube_yaml, environment, local)` pairs the data with its description.

- `environment="TEST"` selects which LINDAS environment the URIs/upload target (`TEST` / `INT` / `PROD`).
- `local=True` keeps everything in-memory and offline — nothing is fetched or uploaded until you ask for it.

Printing the cube shows its target IRI and a preview of the data. At this point the RDF graph is still empty — we have only set things up.

In [ ]:
from pylindas.pycube import Cube

cube = Cube(dataframe=df, cube_yaml=cube_yaml, environment="TEST", local=True)
print(cube)

## 4. Prepare the data (apply the mappings)

`prepare_data()` does two things:

1. **Mints an observation URI per row** from the Key Dimensions. Here that is `Jahr` + `Station`, giving IRIs like `.../observation/2000_Bern`.
2. **Applies each dimension's `mapping`**, turning raw cell values into their final RDF form. In `description.yml`:
   - `Jahr` uses `type: additive` with `base: https://ld.admin.ch/time/year/`, so `2000` becomes the shared year resource `https://ld.admin.ch/time/year/2000`.
   - `Station` uses `type: replace`, mapping `Bern`/`Zürich` to fixed station IRIs.
   - `Wert`, `Standardfehler`, `Status` have no mapping — they stay as literals.

After this call the dataframe is indexed by the observation URI and the mapped columns hold URIs instead of raw strings. The graph is still empty; mapping only reshapes the table.

In [ ]:
cube.prepare_data()
print(cube)

## 5. Write the cube, observations and shape

Now we build the RDF graph in three steps. Watch the triple count grow in each `print(cube)`.

### 5a. `write_cube()` — the `cube:Cube`
Emits the dataset resource and its metadata (name, description, publisher, version, status, contact point, ...) from the cube-level part of `description.yml`.

In [ ]:
cube.write_cube()
print(cube)

### 5b. `write_observations()` — the data points

Emits one `cube:Observation` per row and collects them in the `cube:ObservationSet`. Each observation links its dimension values via the per-dimension properties (built from the `path` fields).

In [ ]:
cube.write_observations()
print(cube)

### 5c. `write_shape()` — the `cube:ObservationConstraint`

Emits the SHACL shape that describes each dimension: its datatype, scale type, unit, and — for nominal dimensions — the list of allowed values (`sh:in`) or the observed min/max for numeric ones. This is what makes the cube self-describing and lets tools like *visualize* render it correctly.

In [ ]:
cube.write_shape()
print(cube)

## 6. Serialize and inspect the Turtle

`serialize(path)` writes the graph to a Turtle (`.ttl`) file. Below we write `mock/cube.ttl` and print the first part so you can see the three resource kinds in one place: the `cube:Cube` with its metadata, the `cube:ObservationSet`, and (further down) the individual observations and the shape.

In [ ]:
TTL_PATH = os.path.join(BASEDIR, "cube.ttl")
cube.serialize(TTL_PATH)

with open(TTL_PATH, encoding="utf-8") as f:
    ttl = f.read()

print(f"Wrote {TTL_PATH} ({len(ttl):,} chars)\n")
print("\n".join(ttl.splitlines()[:48]))

In [ ]:
# Show one observation block to see how a single data point looks in RDF.
for block in ttl.split("\n\n"):
    if "/observation/2000_Bern" in block and " a cube:Observation" in block:
        print(block.strip())
        break

## 7. (Optional) Validate against the `cube.link` SHACL profiles

`cube.validate()` checks the graph against the official `cube.link` SHACL shapes (the standalone cube constraint, plus the visualize profile when `Visualize: True`). It returns `(is_valid, report_text)`.

This fetches the shapes from `cube.link` over the network, so it needs internet access. Set `DO_VALIDATE = True` to run it.

In [ ]:
DO_VALIDATE = False  # requires network access to cube.link

if DO_VALIDATE:
    is_valid, report = cube.validate()
    print("Valid:", is_valid)
    print(report)
else:
    print("Validation skipped (set DO_VALIDATE = True to run it).")

## 8. (Optional) Upload to a triple store

Disabled by default. LINDAS runs on GraphDB; `upload_ttl(...)` reads a `lindas.ini` for the endpoint and credentials of the chosen `environment` and `POST`s the Turtle into the target named graph via the SPARQL 1.1 Graph Store HTTP Protocol. The `lindas.ini` has the form:

```
[TEST]
endpoint=https://graphdb.test.cz-aws.net/repositories/lindas/rdf-graphs/service
username=a-lindas-user-name
password=...
```

Flip `DO_UPLOAD = True` once that file is in place.

In [ ]:
DO_UPLOAD = False  # set to True once lindas.ini is configured

if DO_UPLOAD:
    from pylindas.lindas.upload import upload_ttl

    upload_ttl(
        filename=TTL_PATH,
        db_file="lindas.ini",
        environment="TEST",
        graph_uri="https://mock.ld.admin.ch/mock-example",
    )
    print("Uploaded", TTL_PATH)
else:
    print("Upload skipped (set DO_UPLOAD = True to enable).")